# GTCRN48kSpectral — 48 kHz Neural Speech Enhancement Training & ONNX Export (Google Colab T4)

This notebook reproduces the exact 48 kHz `GTCRN48kSpectral` model (908,663 trainable parameters, 960-point Hann STFT, 480-sample / 10 ms hop, `[1, T, 481, 2]` Complex Ratio Mask) used in **Browser Noise Free Mic**.

- **Clean speech:** 450 native 48 kHz clips from VoiceBank-DEMAND (`yairamr/voicebank-demand-fingerprint-48k`)
- **Real noise:** 120 native 48 kHz environmental noise clips streamed from Microsoft DNS Challenge 5 fullband
- **Synthetic household noise:** Pink-spectrum ceiling fan + 50/100/150 Hz AC mains hum
- **Training:** 25 epochs on Google Colab NVIDIA Tesla T4 GPU (16 GB VRAM), AdamW, 2.0 s segments (282.1 s total, best checkpoint at epoch 20 with loss `0.1081`)
- **Exported weights:** `web/models/gtcrn48k_fp32.onnx` (3,644,893 B / 3.64 MB) and `web/models/gtcrn48k_int8.onnx` (1,342,002 B / 1.34 MB)


In [1]:
import os, re, io, time, random, tarfile, urllib.request, subprocess
from google.colab import drive

drive.mount('/content/drive')
DRIVE_APP_DIR = '/content/drive/MyDrive/Browser Noise Free Mic'
DRIVE_MODELS_DIR = os.path.join(DRIVE_APP_DIR, 'web', 'models')
DRIVE_CKPT_DIR = os.path.join(DRIVE_APP_DIR, 'checkpoints')
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN', '')  # Optional Hugging Face token for faster downloads

# 1B. Train & Export Custom GTCRN-DF 48 kHz on Real 48 kHz VoiceBank-DEMAND Speech + Indian Household Noise
#     (Saves permanently to Browser Noise Free Mic/web/models/gtcrn48k_fp32.onnx & gtcrn48k_int8.onnx)
fp32_drive = os.path.join(DRIVE_MODELS_DIR, "gtcrn48k_fp32.onnx")
int8_drive = os.path.join(DRIVE_MODELS_DIR, "gtcrn48k_int8.onnx")

FORCE_RETRAIN_GTCRN = True  # Set False after your real-speech GTCRN-DF weights are saved
if FORCE_RETRAIN_GTCRN or not (os.path.exists(fp32_drive) and os.path.getsize(fp32_drive) > 500_000 and os.path.exists(int8_drive) and os.path.getsize(int8_drive) > 200_000):
    print("Training & exporting custom 48 kHz GTCRN-DF ONNX on real 48 kHz speech + Indian household noise...")
    subprocess.run(["pip", "install", "-q", "onnx", "onnxruntime", "datasets", "soundfile"], check=True)
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    import onnxruntime as ort
    from onnxruntime.quantization import quantize_dynamic, QuantType

    t0 = time.time()
    use_gpu = torch.cuda.is_available()
    device = torch.device("cuda" if use_gpu else "cpu")
    SR, N_FFT, HOP, N_FREQS = 48000, 960, 480, 481


  [VOICEBANK 48K OK] Loaded 450 native 48 kHz clean speech clips.
  Fetching real 48 kHz noise clips from Microsoft DNS Challenge 5...
  Streaming 48 kHz noise shard from Microsoft DNS5: noise_fullband/datasets_fullband.noise_fullband.audioset_000.tar.bz2 ...
  [MICROSOFT DNS5 48K OK] Streamed 120 real 48 kHz environmental noise clips.
  GTCRN48kSpectral params: 908,663
  Epoch 01/25 | loss=0.2227 | best=0.2227 | elapsed=106.6s
  Epoch 05/25 | loss=0.1354 | best=0.1354 | elapsed=135.4s
  Epoch 10/25 | loss=0.1231 | best=0.1231 | elapsed=171.1s
  Epoch 15/25 | loss=0.1174 | best=0.1163 | elapsed=208.1s
  Epoch 20/25 | loss=0.1081 | best=0.1081 | elapsed=245.0s
  Epoch 25/25 | loss=0.1120 | best=0.1081 | elapsed=282.1s


  [SAVED & VERIFIED IN DRIVE] gtcrn48k_fp32.onnx (3,644,893 bytes)
  [SAVED & VERIFIED IN DRIVE] gtcrn48k_int8.onnx (1,342,002 bytes)
  [VERIFIED IN REPO] web/models/df3/enc.onnx (1,954,042 bytes)
  [VERIFIED IN REPO] web/models/df3/erb_dec.onnx (3,292,397 bytes)
  [VERIFIED IN REPO] web/models/df3/df_dec.onnx (3,340,803 bytes)
  [VERIFIED IN REPO] web/models/gtcrn48k_fp32.onnx (3,644,893 bytes)
  [VERIFIED IN REPO] web/models/gtcrn48k_int8.onnx (1,342,002 bytes)

=== GIT STATUS ===

=== GIT DIFF SUMMARY ===
